In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("WooCommerceOrderStream")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")   # small number - this is a local demo, not a cluster
    .config("spark.ui.showConsoleProgress", "false")  # stops the noisy progress bars in notebook output
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

Spark version: 3.5.5


## Schema
Structured Streaming needs an explicit schema for file sources (it won't infer one on the fly). We only need to list the fields we actually care about — Spark will read those out of each JSON file and ignore everything else (tax lines, meta_data, `_links`, etc.).

In [2]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, ArrayType, DoubleType

billing_schema = StructType([
    StructField("first_name", StringType()),
    StructField("last_name", StringType()),
    StructField("city", StringType()),
    StructField("state", StringType()),
    StructField("postcode", StringType()),
    StructField("country", StringType()),
    StructField("email", StringType()),
    StructField("phone", StringType()),
])

line_item_schema = StructType([
    StructField("product_id", LongType()),
    StructField("name", StringType()),
    StructField("quantity", LongType()),
    StructField("subtotal", StringType()),
    StructField("price", DoubleType()),
])

order_schema = StructType([
    StructField("id", LongType()),
    StructField("status", StringType()),
    StructField("currency", StringType()),
    StructField("date_created_gmt", StringType()),
    StructField("date_modified_gmt", StringType()),
    StructField("total", StringType()),
    StructField("customer_id", LongType()),
    StructField("billing", billing_schema),
    StructField("line_items", ArrayType(line_item_schema)),
])

## Read the landing folder as a stream

In [3]:
LANDING_PATH = "woo_orders_landing"  # same relative folder the poller writes into

raw_stream = (
    spark.readStream
    .schema(order_schema)
    .option("multiLine", True)
    .json(LANDING_PATH)
)

## Clean up: order-level view
Casts `total` from string to a real number, and pulls out billing city/state/country as top-level columns.

In [4]:
from pyspark.sql.functions import col, to_timestamp

order_level = raw_stream.select(
    col("id").alias("order_id"),
    col("status"),
    to_timestamp(col("date_created_gmt")).alias("date_created"),
    to_timestamp(col("date_modified_gmt")).alias("date_modified"),
    col("total").cast("double").alias("total"),
    col("customer_id"),
    col("billing.city").alias("city"),
    col("billing.state").alias("state"),
    col("billing.country").alias("country"),
)

## Clean up: line-item level view
Each order can have multiple products. `explode` turns the `line_items` array into one row per product per order — needed for "best-selling product" type analysis.

In [5]:
from pyspark.sql.functions import explode

line_items_flat = raw_stream.select(
    col("id").alias("order_id"),
    col("status"),
    to_timestamp(col("date_modified_gmt")).alias("date_modified"),
    col("total").cast("double").alias("order_total"),
    explode(col("line_items")).alias("li"),
).select(
    "order_id", "status", "date_modified", "order_total",
    col("li.product_id").alias("product_id"),
    col("li.name").alias("product_name"),
    col("li.quantity").alias("quantity"),
    col("li.subtotal").cast("double").alias("line_subtotal"),
)

## Run the streams into in-memory tables
`format("memory")` keeps results queryable with plain SQL right in this notebook — good for building and checking things before we wire up a real dashboard.

**Notebook note:** this runs for 60 seconds then stops on its own (so the cell doesn't block forever). Bump `timeout` up if you want it watching longer, or set it to `None` and use the kernel's Interrupt button to stop it manually.

In [6]:
order_query = (
    order_level.writeStream
    .format("memory")
    .queryName("orders_flat")
    .outputMode("append")
    .trigger(processingTime="10 seconds")
    .start()
)

line_item_query = (
    line_items_flat.writeStream
    .format("memory")
    .queryName("line_items_flat")
    .outputMode("append")
    .trigger(processingTime="10 seconds")
    .start()
)

timeout = 60
order_query.awaitTermination(timeout)
line_item_query.awaitTermination(timeout)
order_query.stop()
line_item_query.stop()
print("Stopped after", timeout, "seconds.")

Stopped after 60 seconds.


## Check what came through

In [7]:
spark.sql("SELECT * FROM orders_flat ORDER BY date_modified DESC LIMIT 10").show(truncate=False)

+--------+----------+-------------------+-------------------+--------+-----------+-----------+-------------+-------+
|order_id|status    |date_created       |date_modified      |total   |customer_id|city       |state        |country|
+--------+----------+-------------------+-------------------+--------+-----------+-----------+-------------+-------+
|104     |completed |2026-07-26 14:48:05|2026-07-26 14:49:38|5020.24 |29         |Haldia     |Uttar Pradesh|IN     |
|111     |pending   |2026-07-26 14:49:37|2026-07-26 14:49:37|27886.71|31         |Vasai-Virar|Haryana      |IN     |
|110     |pending   |2026-07-26 14:49:25|2026-07-26 14:49:26|33803.77|21         |Bijapur    |Kerala       |IN     |
|100     |completed |2026-07-26 14:46:55|2026-07-26 14:49:13|89606.74|31         |Vasai-Virar|Haryana      |IN     |
|109     |completed |2026-07-26 14:49:09|2026-07-26 14:49:10|92922.55|24         |Anantapuram|Rajasthan    |IN     |
|108     |pending   |2026-07-26 14:48:59|2026-07-26 14:48:59|160

In [8]:
spark.sql("""
    SELECT product_name, COUNT(*) AS times_ordered, SUM(line_subtotal) AS revenue
    FROM line_items_flat
    GROUP BY product_name
    ORDER BY revenue DESC
""").show(truncate=False)

+-------------------------+-------------+------------------+
|product_name             |times_ordered|revenue           |
+-------------------------+-------------+------------------+
|Officia Wireless Earbuds |3            |153846.32         |
|Officiis Baseball Cap    |6            |148729.12         |
|Odit LED Desk Lamp       |5            |130219.92000000001|
|Vel Shampoo Bar          |4            |117143.4          |
|Optio Lip Balm Set       |3            |92059.83          |
|Quae Throw Pillow        |4            |77753.2           |
|Non Makeup Brush Set     |2            |74683.26          |
|Quibusdam Running Shoes  |4            |56896.62          |
|Veniam Laptop Stand      |2            |56201.43000000001 |
|Illum Power Bank         |3            |51513.64          |
|Ducimus Smartwatch       |1            |38429.88          |
|Illum Wool Sweater       |2            |27958.2           |
|Alias Bluetooth Speaker  |3            |21965.72          |
|Nobis Air Fryer        

## Windowed revenue-per-product (the actual "streaming analytics" part)
This is the piece that makes it a *streaming* pipeline rather than a one-off batch query — it buckets orders into 1-hour windows and keeps a running revenue total per product per window, updating as new orders arrive.

**Important:** output mode is `"update"`, not `"append"`. Append mode only emits a window's result once it's fully "closed" (based on the watermark) — for a live dashboard you want to see partial, in-progress totals update as data arrives, which is what `"update"` gives you.

In [9]:
from pyspark.sql.functions import window, sum as _sum, count as _count

product_revenue = (
    line_items_flat
    .withWatermark("date_modified", "10 minutes")
    .groupBy(
        window(col("date_modified"), "1 hour"),
        col("product_name"),
    )
    .agg(
        _sum("line_subtotal").alias("revenue"),
        _count("*").alias("units_sold_lines"),
    )
)

agg_query = (
    product_revenue.writeStream
    .format("memory")
    .queryName("product_revenue")
    .outputMode("update")
    .trigger(processingTime="10 seconds")
    .start()
)

agg_query.awaitTermination(60)
agg_query.stop()
print("Stopped.")

Stopped.


In [10]:
spark.sql("SELECT * FROM product_revenue ORDER BY revenue DESC").show(truncate=False)

+------------------------------------------+-------------------------+------------------+----------------+
|window                                    |product_name             |revenue           |units_sold_lines|
+------------------------------------------+-------------------------+------------------+----------------+
|{2026-07-26 14:30:00, 2026-07-26 15:30:00}|Officia Wireless Earbuds |153846.32         |3               |
|{2026-07-26 14:30:00, 2026-07-26 15:30:00}|Officiis Baseball Cap    |148729.12         |6               |
|{2026-07-26 14:30:00, 2026-07-26 15:30:00}|Odit LED Desk Lamp       |130219.92000000001|5               |
|{2026-07-26 14:30:00, 2026-07-26 15:30:00}|Vel Shampoo Bar          |117143.4          |4               |
|{2026-07-26 14:30:00, 2026-07-26 15:30:00}|Optio Lip Balm Set       |92059.83          |3               |
|{2026-07-26 14:30:00, 2026-07-26 15:30:00}|Quae Throw Pillow        |77753.2           |4               |
|{2026-07-26 14:30:00, 2026-07-26 15: